# 03 · Codebook IEFIC — Encuesta de Ingresos y Gastos (BANREP)

**Dataset:** BANREP-IEFIC-2017-2018.xml — diccionario de variables (DDI) de la Encuesta de Ingresos y Gastos de los Hogares.
**Nota:** este archivo es un **codebook** (diccionario de variables), no contiene microdatos. Documenta 331 variables de la encuesta.

**Utilidad para el proyecto:** la IEFIC es la fuente oficial de **ingresos y gastos de los hogares colombianos** — insumo clave para caracterizar la capacidad de pago de los solicitantes de crédito.

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "..")
from src.utils.loaders import load_iefic_codebook

cb = load_iefic_codebook()
print(f"Variables únicas: {cb.shape[0]}")
cb.head()

Variables únicas: 331


,name,label,question,type
0,SECUENCIA_P,secuencia_p,SECUENCIA_P,discrete
1,ORDEN,orden,ORDEN,discrete
2,DIRECTORIO,DIRECTORIO,,contin
3,INGRESO_COMPLETO,"estado del ingreso total 1=completo, 0=falta a...","Estado del ingreso total 1=completo, 0=Falta a...",discrete
4,P6050,¿cuál es el parentesco de ... con el jefe o je...,¿Cuál es el parentesco de ... Con el jefe o je...,discrete


In [2]:
print("Tipo de variable:")
print(cb["type"].value_counts().to_string())
print("\nVariables con pregunta documentada:", cb["question"].str.len().gt(0).sum())

Tipo de variable:
type
discrete    206
contin      125

Variables con pregunta documentada: 328


## Variables relevantes para riesgo de crédito

Buscamos variables de **ingresos, deuda, empleo y patrimonio** — los factores clásicos de capacidad de pago.

In [3]:
keywords = ["ingreso", "deuda", "crédito", "credito", "empleo", "trabajo", "patrimonio",
             "ahorro", "gasto", "ocupaci", "salario", "pension", "arriendo", "vivienda"]
mask = cb["label"].str.lower().str.contains("|".join(keywords), na=False) | \
       cb["question"].str.lower().str.contains("|".join(keywords), na=False)
relevantes = cb[mask].copy()
print(f"Variables potencialmente relevantes: {relevantes.shape[0]}")
relevantes[["name", "label", "type"]].head(30)

Variables potencialmente relevantes: 143


,name,label,type
3,INGRESO_COMPLETO,"estado del ingreso total 1=completo, 0=falta a...",discrete
6,INGTOTOB,ingreso total por persona,contin
10,P2439,¿algún miembro de este hogar es propietario de...,discrete
11,P2447,¿en qué año ud. o algún miembro de su hogar c...,contin
12,P2461,si usted quisiera vender esta vivienda ¿cuál s...,contin
13,P2462,"ud. o algún miembro de su hogar, ¿compró o con...",discrete
15,P2465,¿este subsidio correspondía a vivienda de int...,discrete
16,P2466,¿para la compra de esta vivienda utilizó crédi...,discrete
17,P2168,valor del crédito hipotecario $,contin
18,P2469,¿ud. o algún miembro del hogar conoce o conocí...,discrete


## Variables de ingreso total y componentes

In [4]:
ingreso = cb[cb["name"].str.contains("ING|P60|P61|P62|P63", na=False)]
ingreso[["name", "label", "type"]].head(20)

,name,label,type
3,INGRESO_COMPLETO,"estado del ingreso total 1=completo, 0=falta a...",discrete
4,P6050,¿cuál es el parentesco de ... con el jefe o je...,discrete
6,INGTOTOB,ingreso total por persona,contin
106,P622,¿………….tiene fondos mutuos o de inversión?,discrete


## Variables de deuda y crédito

In [5]:
deuda = cb[cb["label"].str.contains("deuda|crédito|credito|financi", case=False, na=False)]
deuda[["name", "label", "type"]].head(20)

,name,label,type
16,P2466,¿para la compra de esta vivienda utilizó crédi...,discrete
17,P2168,valor del crédito hipotecario $,contin
18,P2469,¿ud. o algún miembro del hogar conoce o conocí...,discrete
19,P2470,¿está pagando este crédito hipotecario actualm...,discrete
20,P2471_1,¿cuánto dinero paga (o debería estar pagando a...,contin
21,P2471_2,¿cuánto dinero paga (o debería estar pagando a...,contin
22,P2471_3,¿cuánto dinero paga (o debería estar pagando a...,contin
23,P2471_4,¿cuánto dinero paga (o debería estar pagando a...,contin
24,P2472,¿con qué tipo de institución tomó el crédito h...,discrete
26,P2473,ud. o algún miembro del hogar ¿tiene una deuda...,discrete


### Hallazgos
- El codebook documenta **331 variables** (412 discretas + 250 continuas en el XML crudo, deduplicadas por archivo F17/F18).
- Existen variables de **ingreso total** (INGRESO_COMPLETO, INGTOTOB) y componentes (P6050 parentesco, P60xx ingresos laborales).
- La IEFIC complementa los datasets de cartera: permite contextualizar la **capacidad de pago** de los hogares, aunque no hay microdatos en este archivo.